诺道医学编程笔试

In [ ]:
print("Git Test")

In [ ]:
print("20260614测试")

In [5]:
import numpy as np 
import pandas as pd
from datetime import datetime, timedelta, time

In [6]:
df = pd.read_excel('nuodao2.xlsx')
df

C:\Users\lenovo\AppData\Local\Temp\ipykernel_13248\598479399.py:1: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  df = pd.read_excel('nuodao2.xlsx')


,PATIENT_ID,ORDER_CONTENT,DOSAGE,FREQUENCY,Unnamed: 4,START_DATETIME,END_DATETIME
0,1048945,环孢素注射液 ...,240,1,一天一次,2014-10-01 08:00:00,2014-10-03 07:00:00
1,1048945,环孢素注射液 ...,210,1,一天一次,2014-10-03 08:00:00,2014-10-05 15:04:48
2,1048945,环孢素注射液 ...,175,2,一天两次,2014-10-06 08:00:00,2014-10-08 11:19:43
3,1048945,环孢素注射液 ...,200,2,一天两次,2014-10-08 13:00:00,2014-10-10 11:32:38
4,1048945,环孢素注射液 ...,200,2,一天两次,2014-10-15 10:00:00,2014-10-21 11:32:38
5,1048945,环孢素注射液 ...,75,3,一天三次,2014-10-21 12:00:00,2014-10-24 11:32:38
6,1056522,环孢素注射液 ...,75,2,一天两次,2014-10-21 08:00:00,2014-10-24 13:33:13
7,1056522,环孢素注射液 ...,50,2,一天两次,2014-10-24 14:00:00,2014-10-25 06:45:08
8,1056522,环孢素注射液 ...,75,2,一天两次,2014-10-25 08:00:00,2014-10-27 08:45:08


In [7]:
df['START_DATETIME'] = pd.to_datetime(df['START_DATETIME'])
df['END_DATETIME'] = pd.to_datetime(df['END_DATETIME'])

In [8]:
# 创建给药时间列表
df['DOSE_TIMES'] = df['FREQUENCY'].apply(lambda x: 
    [time(12,0)] if x == 1 else 
    [time(8,0), time(18,0)] if x == 2 else 
    [time(8,0), time(12,0), time(18,0)])

# 展开DataFrame
df_exploded = df.explode('DOSE_TIMES')

# 创建完整的日期时间
def create_dose_datetimes(row):
    date_range = pd.date_range(start=row['START_DATETIME'].date(), end=row['END_DATETIME'].date())
    return [datetime.combine(date, row['DOSE_TIMES']) for date in date_range]

df_exploded['DOSE_DATETIME'] = df_exploded.apply(create_dose_datetimes, axis=1)
df_exploded = df_exploded.explode('DOSE_DATETIME')

# 筛选在开始和结束时间范围内的记录
df_exploded = df_exploded[
    (df_exploded['DOSE_DATETIME'] >= df_exploded['START_DATETIME']) & 
    (df_exploded['DOSE_DATETIME'] <= df_exploded['END_DATETIME'])
]

# 按病人ID和日期分组，计算每日总剂量
result = df_exploded.groupby(['PATIENT_ID', df_exploded['DOSE_DATETIME'].dt.date])['DOSAGE'].sum().reset_index()
result.columns = ['PATIENT_ID', 'time', '日剂量']

# 添加ORDER_CONTENT列
result = result.merge(df[['PATIENT_ID', 'ORDER_CONTENT']].drop_duplicates(), on='PATIENT_ID', how='left')

# 整理列的顺序
result = result[['PATIENT_ID', 'ORDER_CONTENT', '日剂量', 'time']]

# 按日期排序
result = result.sort_values('time').reset_index(drop=True)

# 显示结果
print(result)

# 保存到Excel文件
result.to_excel('result2.xlsx', index=False)

    PATIENT_ID                                      ORDER_CONTENT  日剂量  \
0      1048945  环孢素注射液                                        ...  240   
1      1048945  环孢素注射液                                        ...  240   
2      1048945  环孢素注射液                                        ...  210   
3      1048945  环孢素注射液                                        ...  210   
4      1048945  环孢素注射液                                        ...  210   
5      1048945  环孢素注射液                                        ...  350   
6      1048945  环孢素注射液                                        ...  350   
7      1048945  环孢素注射液                                        ...  375   
8      1048945  环孢素注射液                                        ...  400   
9      1048945  环孢素注射液                                        ...  200   
10     1048945  环孢素注射液                                        ...  200   
11     1048945  环孢素注射液                                        ...  400   
12     1048945  环孢素注射液                